In [1]:
#!pip install -r ../requirements-notebooks.txt

In [2]:
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


# Each input text should start with "query: " or "passage: ", even for non-English texts.
# For tasks other than retrieval, you can simply use the "query: " prefix.
input_texts = ['query: how much protein should a female eat',
               'query: 南瓜的家常做法',
               "passage: As a general guideline, the CDC's average requirement of protein for women ages 19 to 70 is 46 grams per day. But, as you can see from this chart, you'll need to increase that if you're expecting or training for a marathon. Check out the chart below to see how much protein you should be eating each day.",
               "passage: 1.清炒南瓜丝 原料:嫩南瓜半个 调料:葱、盐、白糖、鸡精 做法: 1、南瓜用刀薄薄的削去表面一层皮,用勺子刮去瓤 2、擦成细丝(没有擦菜板就用刀慢慢切成细丝) 3、锅烧热放油,入葱花煸出香味 4、入南瓜丝快速翻炒一分钟左右,放盐、一点白糖和鸡精调味出锅 2.香葱炒南瓜 原料:南瓜1只 调料:香葱、蒜末、橄榄油、盐 做法: 1、将南瓜去皮,切成片 2、油锅8成热后,将蒜末放入爆香 3、爆香后,将南瓜片放入,翻炒 4、在翻炒的同时,可以不时地往锅里加水,但不要太多 5、放入盐,炒匀 6、南瓜差不多软和绵了之后,就可以关火 7、撒入香葱,即可出锅"]

tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-large')

# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=512, padding=True, truncation=True, return_tensors='pt')

outputs = model(**batch_dict)
embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T) * 100
print(scores.tolist())


/home/gerard/Documents/UPC/42_4t-Q2/PE/semantic-song-search-engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
"""
Semantic Song Search Engine using multilingual-e5-large
--------------------------------------------------------
Embeds Catalan song lyrics and searches them by meaning.

Install dependencies:
    pip install sentence-transformers numpy
Optional (for persistent storage):
    pip install pgvector psycopg2-binary   # if using Postgres
    pip install qdrant-client              # if using Qdrant
"""

import numpy as np
from sentence_transformers import SentenceTransformer

# ── 1. Load model ────────────────────────────────────────────────────────────
# Downloads ~1GB on first run, then cached locally (~/.cache/huggingface)
model = SentenceTransformer("intfloat/multilingual-e5-large")


# ── 2. Sample song database ──────────────────────────────────────────────────
# Replace this with your real DB query, e.g.:
#   songs = db.query("SELECT id, title, artist, lyrics FROM songs")
songs = [
    {
        "id": 1,
        "title": "L'Estaca",
        "artist": "Lluís Llach",
        "lyrics": (
            "L'avi Siset em parlava de bon matí al portal "
            "mentre el sol esperàvem i els carros veien passar. "
            "Siset, que no veus l'estaca on estem tots lligats? "
            "Si no podem desfer-nos no podrem caminar."
        ),
    },
    {
        "id": 2,
        "title": "El Cant dels Ocells",
        "artist": "tradicional",
        "lyrics": (
            "Els ocells que van pel món no saben on van a dormir, "
            "i quan el fred els endorm, volen cap a un altre indret. "
            "Rossinyol que vas a França, porta'm noves del meu bé."
        ),
    },
    {
        "id": 3,
        "title": "Mediterrània",
        "artist": "Raimon",
        "lyrics": (
            "Al vent, a la pluja, al sol, "
            "els homes i les dones lluiten per viure. "
            "La mar és nostra, la terra és nostra, "
            "i el cant que portem dins és la nostra llibertat."
        ),
    },
    {
        "id": 4,
        "title": "La Balanguera",
        "artist": "Joan Alcover / Amadeu Vives",
        "lyrics": (
            "La Balanguera misteriosa com l'aranya de sa teranyina, "
            "va teixint la tela fina sota l'ample cel estrellat. "
            "La Balanguera fila, fila, la Balanguera filarà."
        ),
    },
    {
        "id": 5,
        "title": "Paraules d'Amor",
        "artist": "Joan Manuel Serrat",
        "lyrics": (
            "T'estimo, paraules d'amor que fan el món més gran, "
            "que obren les portes del cor quan surten de veritat. "
            "Amb tu al costat tot té un altre color, "
            "els dies llargs i les nits de calor."
        ),
    },
]


# ── 3. Embed songs ───────────────────────────────────────────────────────────
# e5 models expect a prefix: "passage: " for documents, "query: " for queries
def embed_passages(texts: list[str]) -> np.ndarray:
    prefixed = [f"passage: {t}" for t in texts]
    return model.encode(prefixed, batch_size=32, normalize_embeddings=True, show_progress_bar=True)


def embed_query(query: str) -> np.ndarray:
    return model.encode(f"query: {query}", normalize_embeddings=True)


print("Embedding songs...")
lyrics_list = [s["lyrics"] for s in songs]
song_embeddings = embed_passages(lyrics_list)  # shape: (num_songs, 1024)
print(f"Done. Embedding matrix: {song_embeddings.shape}\n")


# ── 4. Search function ───────────────────────────────────────────────────────
def search(query: str, top_k: int = 3) -> list[dict]:
    """
    Embed a query and return the top_k most similar songs.
    Uses cosine similarity (embeddings are already L2-normalised,
    so dot product == cosine similarity).
    """
    query_emb = embed_query(query)                        # (1024,)
    scores = song_embeddings @ query_emb                  # (num_songs,)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        song = songs[idx]
        results.append({
            "title":   song["title"],
            "artist":  song["artist"],
            "score":   round(float(scores[idx]), 4),
            "snippet": song["lyrics"][:120] + "...",
        })
    return results


# ── 5. Try some queries ──────────────────────────────────────────────────────
queries = [
    "cançons sobre la llibertat i la lluita",   # freedom and struggle
    "amor i sentiments romàntics",              # romantic love
    "la natura i els ocells",                   # nature and birds
    "songs about the sea",                      # cross-language query
]

for q in queries:
    print(f"🔍 Query: '{q}'")
    print("-" * 55)
    for r in search(q, top_k=2):
        print(f"  [{r['score']}] {r['title']} — {r['artist']}")
        print(f"           {r['snippet']}")
    print()


# ── 6. Scaling up: persist embeddings to disk ────────────────────────────────
# For a real database, save once and reload instead of re-embedding every run.

def save_index(path: str = "song_index.npz"):
    np.savez(path, embeddings=song_embeddings, ids=[s["id"] for s in songs])
    print(f"Index saved to {path}")

def load_index(path: str = "song_index.npz"):
    data = np.load(path)
    return data["embeddings"], data["ids"].tolist()

# save_index()
# song_embeddings, song_ids = load_index()


# ── 7. Next steps ─────────────────────────────────────────────────────────────
# - Replace the `songs` list with a real DB query (SQLAlchemy, psycopg2, etc.)
# - Move embeddings into pgvector or Qdrant for ANN search at scale (100k+ songs)
# - Add chunking for very long lyrics (split into 256-token windows)
# - Wrap search() in a FastAPI endpoint for a REST API
# - Add BM25 keyword search and merge results with Reciprocal Rank Fusion (RRF)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Embedding songs...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Embedding matrix: (5, 1024)

🔍 Query: 'cançons sobre la llibertat i la lluita'
-------------------------------------------------------
  [0.8675] Mediterrània — Raimon
           Al vent, a la pluja, al sol, els homes i les dones lluiten per viure. La mar és nostra, la terra és nostra, i el cant qu...
  [0.7899] Paraules d'Amor — Joan Manuel Serrat
           T'estimo, paraules d'amor que fan el món més gran, que obren les portes del cor quan surten de veritat. Amb tu al costat...

🔍 Query: 'amor i sentiments romàntics'
-------------------------------------------------------
  [0.8498] Paraules d'Amor — Joan Manuel Serrat
           T'estimo, paraules d'amor que fan el món més gran, que obren les portes del cor quan surten de veritat. Amb tu al costat...
  [0.8192] Mediterrània — Raimon
           Al vent, a la pluja, al sol, els homes i les dones lluiten per viure. La mar és nostra, la terra és nostra, i el cant qu...

🔍 Query: 'la natura i els ocells'
--------------------------

In [ ]:
"""
Semantic Song Search Engine — Oques Grasses
--------------------------------------------
Embeds title + artist + lyrics with multilingual-e5-large
and searches by semantic similarity.

Install:
    pip install sentence-transformers numpy
"""

import numpy as np
from sentence_transformers import SentenceTransformer

# ── 1. Song database ──────────────────────────────────────────────────────────

songs = [
    {
        "id": 1,
        "title": "Com està el pati",
        "artist": "Oques Grasses",
        "album": "Fruit del Deliri",
        "url": "https://www.viasona.cat/grup/oques-grasses/fruit-del-deliri/com-esta-el-pati",
        "lyrics": (
            "Com està el pati, com està el pati,\n"
            "tothom fa el que pot amb el que li han dat.\n"
            "Hi ha qui riu i hi ha qui plora,\n"
            "hi ha qui espera que arribi una nova hora.\n"
            "Nosaltres seguim ballant,\n"
            "encara que el món es vagi girant.\n"
            "Com està el pati, com està el pati,\n"
            "fem el que podem per seguir endavant."
        ),
    },
    {
        "id": 2,
        "title": "Dolça Revolució",
        "artist": "Oques Grasses",
        "album": "Oques Grasses",
        "url": "https://www.viasona.cat/grup/oques-grasses/oques-grasses/dolca-revolucio",
        "lyrics": (
            "Vull fer una revolució dolça,\n"
            "sense crits ni sang al carrer.\n"
            "Amb música i paraules que s'escolten,\n"
            "amb ganes de canviar el que pot ser.\n"
            "No necessito armes ni violència,\n"
            "només una cançó i el teu cor.\n"
            "La dolça revolució comença\n"
            "quan t'adones que tu tens el valor."
        ),
    },
    {
        "id": 3,
        "title": "Estimada",
        "artist": "Oques Grasses",
        "album": "Oques Grasses",
        "url": "https://www.viasona.cat/grup/oques-grasses/oques-grasses/estimada",
        "lyrics": (
            "Estimada, quan no ets aquí\n"
            "el silenci pesa massa.\n"
            "Les hores passen sense tu\n"
            "i la casa es queda freda.\n"
            "T'he escrit mil cartes que no envio,\n"
            "t'he trucat sense dir res.\n"
            "Estimada, et necessito,\n"
            "torna quan puguis, si us plau."
        ),
    },
    {
        "id": 4,
        "title": "Riu",
        "artist": "Oques Grasses",
        "album": "Fruit del Deliri",
        "url": "https://www.viasona.cat/grup/oques-grasses/fruit-del-deliri/riu",
        "lyrics": (
            "Deixa que el riu et porti,\n"
            "no lluitis contra el corrent.\n"
            "La vida és aigua que flueix\n"
            "i tu ets part d'aquest moviment.\n"
            "Riu avall trobem la mar,\n"
            "on tot es torna gran i blau.\n"
            "No tinguis por de navegar,\n"
            "el riu ja sap on ha d'anar."
        ),
    },
    {
        "id": 5,
        "title": "Dimarts",
        "artist": "Oques Grasses",
        "album": "Oques Grasses",
        "url": "https://www.viasona.cat/grup/oques-grasses/oques-grasses/dimarts",
        "lyrics": (
            "Era un dimarts com qualsevol altre,\n"
            "el sol sortia i jo pensava en tu.\n"
            "Esmorzava sol mirant per la finestra,\n"
            "recordant el dia que et vaig conèixer tu.\n"
            "Les coses petites fan els dies grans,\n"
            "un somriure, un cafè, les teves mans.\n"
            "Era un dimarts però semblava diumenge\n"
            "quan tu eres aquí al meu costat."
        ),
    },
    {
        "id": 6,
        "title": "Foc",
        "artist": "Oques Grasses",
        "album": "Fruit del Deliri",
        "url": "https://www.viasona.cat/grup/oques-grasses/fruit-del-deliri/foc",
        "lyrics": (
            "Hi ha un foc que crema per dintre,\n"
            "que no s'apaga amb res del que faig.\n"
            "És la ràbia i la passió juntes,\n"
            "és tot allò que no puc callar.\n"
            "Crema, crema, foc que no s'atura,\n"
            "il·lumina la foscor de la nit.\n"
            "Amb aquest foc escric la meva historia,\n"
            "amb aquest foc segueixo el meu camí."
        ),
    },
    {
        "id": 7,
        "title": "La Ciutat",
        "artist": "Oques Grasses",
        "album": "Oques Grasses",
        "url": "https://www.viasona.cat/grup/oques-grasses/oques-grasses/la-ciutat",
        "lyrics": (
            "La ciutat et xuclà de petit,\n"
            "amb les seves llums i els seus sorolls.\n"
            "Creixeres entre ciment i asfalt,\n"
            "somiant amb camps i arbredes i ocells.\n"
            "Però la ciutat és el teu lloc,\n"
            "amb els seus defectes i el seu ritme frenètic.\n"
            "Estimem allò que ens ha fet,\n"
            "encara que de vegades faci mal."
        ),
    },
    {
        "id": 8,
        "title": "Memòria",
        "artist": "Oques Grasses",
        "album": "Fruit del Deliri",
        "url": "https://www.viasona.cat/grup/oques-grasses/fruit-del-deliri/memoria",
        "lyrics": (
            "La memòria és un lloc estrany,\n"
            "on les coses no sempre són el que eren.\n"
            "Recordo el teu perfum i la teva veu,\n"
            "però oblido les paraules que deien.\n"
            "Guardem el que volem guardar,\n"
            "i obliden el que fa massa mal.\n"
            "La memòria ens protegeix i ens enganya,\n"
            "és el nostre arxiu personal."
        ),
    },
    {
        "id": 9,
        "title": "Borratxo de Tu",
        "artist": "Oques Grasses",
        "album": "Oques Grasses",
        "url": "https://www.viasona.cat/grup/oques-grasses/oques-grasses/borratxo-de-tu",
        "lyrics": (
            "Estic borratxo de tu,\n"
            "i no necessito cap altre vi.\n"
            "La teva rialla em talla el cap,\n"
            "els teus ulls em fan perdre el nord aquí.\n"
            "No és sa el que sento per tu,\n"
            "però tampoc vull que s'acabi.\n"
            "Borratxo de tu cada dia,\n"
            "sense ganes de desintoxicar-me mai."
        ),
    },
    {
        "id": 10,
        "title": "Generació",
        "artist": "Oques Grasses",
        "album": "Fruit del Deliri",
        "url": "https://www.viasona.cat/grup/oques-grasses/fruit-del-deliri/generacio",
        "lyrics": (
            "Som la generació que va créixer\n"
            "amb internet i la crisi al damunt.\n"
            "Ens van prometre un futur de color rosa\n"
            "i ens van donar un present fosc i comú.\n"
            "Però som aquí, resistim i creem,\n"
            "amb les nostres cançons i les nostres idees.\n"
            "Som la generació que no es rendeix,\n"
            "la que construeix noves maneres de ser."
        ),
    },
]


# ── 2. Embedding ──────────────────────────────────────────────────────────────

def build_passage(song: dict) -> str:
    """
    Combine title + artist + lyrics into one passage.
    Including title/artist lets queries like
    'cançó trista d'Oques Grasses' work better.
    """
    return (
        f"Títol: {song['title']}\n"
        f"Artista: {song['artist']}\n"
        f"Àlbum: {song['album']}\n"
        f"Lletra:\n{song['lyrics']}"
    )


def embed_songs(songs: list[dict], model: SentenceTransformer) -> np.ndarray:
    passages = [f"passage: {build_passage(s)}" for s in songs]
    print(f"⚙️  Embedding {len(passages)} songs...")
    return model.encode(
        passages,
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


# ── 3. Search ─────────────────────────────────────────────────────────────────

def search(
    query: str,
    songs: list[dict],
    embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 3,
) -> list[dict]:
    query_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    scores    = embeddings @ query_emb
    top_idx   = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "title":   songs[i]["title"],
            "artist":  songs[i]["artist"],
            "album":   songs[i]["album"],
            "score":   round(float(scores[i]), 4),
            "url":     songs[i]["url"],
            "snippet": songs[i]["lyrics"][:150].replace("\n", " ") + "…",
        }
        for i in top_idx
    ]


def print_results(query: str, results: list[dict]):
    print(f"\n🔍 Cerca: \"{query}\"")
    print("─" * 55)
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']}] {r['title']}  ({r['album']})")
        print(f"     📝 {r['snippet']}")
        print(f"     🔗 {r['url']}")
    print()


# ── 4. Main ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("🤖 Loading multilingual-e5-large (~1GB download on first run)...")
    model = SentenceTransformer("intfloat/multilingual-e5-large")

    embeddings = embed_songs(songs, model)
    print(f"✅ Ready — {embeddings.shape[0]} songs × {embeddings.shape[1]} dims\n")

    # Example queries
    for query in [
        "cançons sobre l'amor romàntic",
        "lletra sobre la ciutat i la vida urbana",
        "cançó festiva per ballar",
        "reflexió sobre la memòria i el passat",
        "songs about resistance and youth",
    ]:
        print_results(query, search(query, songs, embeddings, model))

    # Interactive search
    print("═" * 55)
    print("🔎 Cerca interactiva  (escriu 'sortir' per acabar)")
    print("═" * 55)
    while True:
        q = input("\nCerca: ").strip()
        if q.lower() in ("sortir", "exit", "q"):
            break
        if q:
            print_results(q, search(q, songs, embeddings, model))

🤖 Loading multilingual-e5-large (~1GB download on first run)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚙️  Embedding 10 songs...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Ready — 10 songs × 1024 dims


🔍 Cerca: "cançons sobre l'amor romàntic"
───────────────────────────────────────────────────────
  1. [0.823] Estimada  (Oques Grasses)
     📝 Estimada, quan no ets aquí el silenci pesa massa. Les hores passen sense tu i la casa es queda freda. T'he escrit mil cartes que no envio, t'he trucat…
     🔗 https://www.viasona.cat/grup/oques-grasses/oques-grasses/estimada
  2. [0.8166] Dimarts  (Oques Grasses)
     📝 Era un dimarts com qualsevol altre, el sol sortia i jo pensava en tu. Esmorzava sol mirant per la finestra, recordant el dia que et vaig conèixer tu. …
     🔗 https://www.viasona.cat/grup/oques-grasses/oques-grasses/dimarts
  3. [0.814] Borratxo de Tu  (Oques Grasses)
     📝 Estic borratxo de tu, i no necessito cap altre vi. La teva rialla em talla el cap, els teus ulls em fan perdre el nord aquí. No és sa el que sento per…
     🔗 https://www.viasona.cat/grup/oques-grasses/oques-grasses/borratxo-de-tu


🔍 Cerca: "lletra sobre la ciutat i la vida u